# Full experiment — Drag-and-drop MUSHRA — configured by one file

> **Full experiment.** Shows a **welcome** screen, then the **consent**
> questionnaire (it sets the anonymous `participant_id`), then the experiment,
> then a **thank-you** screen. Every saved file is named with that participant id.


This notebook runs a **drag-and-drop, MUSHRA-like rating** experiment and saves
the ratings. On each screen the participant hears a **reference** and several
**test** stimuli and rates each one by dragging its tile along a scale.

Like the other whispy experiments it is **config-driven**: the whole experiment
is described in one file,
[`configs/drag_and_drop_mushra.yml`](../configs/drag_and_drop_mushra.yml); only
the shared look comes from [`configs/design.yml`](../configs/design.yml). You
should not need to edit any Python below.

That one file is split into four blocks, each picked up by the consumer that
needs it:

| block in the YAML | used by | configures |
|---|---|---|
| `output:` + `SoundDevice:` / `OSCHandler:` | the stimuli handler | whether a screen plays a WAV or sends an OSC message per stimulus id |
| `ui:` | the MUSHRA window | window size, rating-area size, autoplay |
| `attributes:` | the MUSHRA window | the rating scales (task text, values, labels) |
| `experiment:` | the scheduler | the blocks/sections and the conditions to rate |

The MUSHRA window reads both the `ui:` and `attributes:` blocks from the single
config you hand it. Set `output: osc` in the config to send OSC messages instead
of playing audio (e.g. to trigger cues in Max/Pd/SuperCollider) — see the
`OSCHandler:` block there; `build_stimuli_handler` and the MUSHRA window code
below are identical either way. If you haven't installed the project yet, run
this from the repo root: `pip install -e .`

## Welcome the participant

Run this first. It greets the participant with a **welcome screen** before
anything else; its message lives in [`configs/welcome.yml`](../configs/welcome.yml)
(markdown supported), so you can reword it without editing the Python. The cell
blocks until the participant clicks **Continue**.

In [ ]:
from pathlib import Path
from whispy.ui import InfoWindow
from whispy.utils import read_config

# The welcome message lives in configs/welcome.yml (markdown supported); reword
# it there without editing this cell.
welcome = read_config(Path('..') / 'configs' / 'welcome.yml')['ui']
InfoWindow(welcome['message'], fullscreen=welcome.get('fullscreen', True), blocking=True)

## Consent

Next, record consent: the cell below shows the consent questionnaire
([`configs/questionnaires/questionnaire_consent.yml`](../configs/questionnaires/questionnaire_consent.yml)),
which records consent and builds an **anonymous participant id** from the
`pid_1..pid_4` answers. The id is kept in the `participant_id` variable and used
to name every result file saved in this notebook, so all of this participant's
data shares one id.

In [ ]:
from pathlib import Path
from whispy.ui import Questionnaire
from whispy.utils import participant_id_from_consent, save_results

consent_config = Path('..') / 'configs' / 'questionnaires' / 'questionnaire_consent.yml'
consent = Questionnaire(questionnaire=consent_config, blocking=True)
consent_results = consent.get_results()
consent.close()

# Anonymous participant id (e.g. 'HPo1'); names every result file below.
participant_id = participant_id_from_consent(consent_results)
print('participant id:', participant_id)
save_results(consent_results, 'consent', participant_id=participant_id)

In [ ]:
import whispy
import pandas as pd
from pathlib import Path
from whispy.interfaces import build_stimuli_handler
from whispy.utils import read_config

config_path = Path('..') / 'configs' / 'drag_and_drop_mushra.yml'   # <- the one file you edit
stimuli_dir = Path('demo_stimuli/mushra')                           # <- folder with your WAVs

cfg = read_config(config_path)
# reads `output:` (sounddevice/osc) and builds the matching handler - swap
# `output: sounddevice` <-> `output: osc` in the config, nothing here changes.
handler = build_stimuli_handler(config_path, stimuli_dir, loop=True)
print('output backend:', cfg.get('output', 'sounddevice'))
print('available stimulus ids:', list(handler.stimuli.keys()))

## Build your own experiment

The WAVs in `examples/demo_stimuli/mushra/` and the design shipped in the config
are **only a demo** (run
`python examples/demo_stimuli/mushra/generate_mushra_stimuli.py` to regenerate
them). To build your own experiment you only edit the config and `stimuli_dir`,
never the Python:

1. **Point `stimuli_dir`** (in the setup cell) at the folder holding your WAVs.
2. **Map ids to files** under `SoundDevice:`, define your rating scales under
   `attributes:`, and lay out the course under `experiment:` using those ids and
   attribute names:
   ```yaml
   SoundDevice:
     ref: { file: reference.wav }
     a:   { file: condition_a.wav }
     b:   { file: condition_b.wav }
   attributes:
     quality:
       task: |
         Rate the
         **Quality**
       labels: [bad, null, null, null, excellent]
       values: [0, .25, .5, .75, 1]
       neutral_value: 0
   experiment:
     - block:
         - block_name: My block
         - section:
             section_name: My section
             attribute: quality      # must match an attribute above
             reference: ref          # must be defined under SoundDevice
             test: [a, b]            # the conditions to rate
   ```

**Safety constraints** (checked when the handler loads in the setup cell — a
violation raises a `ValueError`):

- **No clipping** — every sample must satisfy `|amplitude| < 1` (normalise with headroom, e.g. ~0.7).
- **One sampling rate** — every file must share the same sampling rate.
- **All ids must exist** — every id used under `experiment:` must be defined under `SoundDevice:`.

Run the next cell to **pre-flight-check your files before testing a participant**.
After changing the config or `stimuli_dir`, re-run the setup cell.

> The pre-flight check below (`check_stimuli`) validates the `SoundDevice:` WAVs
> and only applies with `output: sounddevice`. With `output: osc` there are no
> files to check — instead make sure every id used under `experiment:` has a
> matching entry under `OSCHandler.stimuli:` in the config.

In [ ]:
import numpy as np
import pyfar as pf

def check_stimuli(config_path, stimuli_dir):
    """Validate the stimulus set: existence, clipping, one sampling rate, and
    that every id used by the experiment is defined."""
    cfg = read_config(config_path)
    sound = cfg.get('SoundDevice', {})
    folder = Path(stimuli_dir)

    # every id referenced by the experiment (reference + test conditions)
    used = set()
    for block in cfg.get('experiment', []):
        for it in block.get('block', []):
            if 'section' in it:
                s = it['section']
                used.add(s.get('reference'))
                used.update(s.get('test', []))

    problems, rates = [], {}
    print(f'Checking {len(sound)} stimuli in {folder.resolve()}\n')
    for sid, spec in sound.items():
        path = folder / spec['file']
        if not path.exists():
            problems.append(f"id {sid!r}: file not found -> {path}")
            print(f"  {str(sid):>8}  MISSING   {spec['file']}")
            continue
        sig = pf.io.read_audio(path)
        peak = float(np.max(np.abs(sig.time)))
        rates[sid] = sig.sampling_rate
        flag = '  <-- CLIPPING' if peak >= 1 else ''
        print(f"  {str(sid):>8}  {sig.sampling_rate} Hz  peak={peak:.3f}  {spec['file']}{flag}")
        if peak >= 1:
            problems.append(f"id {sid!r}: clips (peak={peak:.3f} >= 1)")

    for rid in used:
        if rid not in sound:
            problems.append(f"experiment uses id {rid!r}, not defined under SoundDevice:")

    if len(set(rates.values())) > 1:
        problems.append(f"mixed sampling rates: {sorted(set(rates.values()))} (all must match)")

    print()
    if problems:
        print('NOT ready - fix these before running:')
        for prob in problems:
            print('  -', prob)
    else:
        print('All good: files exist, no clipping, one sampling rate, all experiment ids defined.')
    return not problems

check_stimuli(config_path, stimuli_dir)

## Schedule the experiment

`ExperimentScheduler` reads the `experiment:` block and turns it into a
randomized sequence of rating screens. By default blocks, sections, and the
conditions within a section are all shuffled (pass `random_seed=...` for a
reproducible order, or `randomize_blocks` / `randomize_sections` /
`randomize_conditions=False` to keep the config order). The cell below shows the
resulting order.

In [ ]:
schedule = whispy.ExperimentScheduler(experiment=cfg['experiment'])

for screen in schedule:
    print(screen['block_name'], '|', screen['section_name'],
          '| attribute:', screen['attribute'],
          '| test:', [int(t) for t in screen['test']])

## Run the experiment

The loop below presents every screen in turn, reusing **one window** for the
whole experiment (the first screen opens it; later screens swap their content
into the same window, so it stays put with no reload). Whenever a new block
starts an info screen announces the next attribute, then the rating window shows
the conditions: the participant clicks each tile to **hear** that stimulus,
plays the **reference** on demand with the **Reference (R)** button or the **R**
key, and **drags** each tile along the scale to rate it. Continue is enabled
once every stimulus (including the reference) has been heard.

After the last screen the window is **closed automatically**, so it is clear to
everyone that the experiment is over. The ratings are accumulated into one
`results` table.

In [ ]:
# accumulate the results across all screens, reusing one window
results = None
host = None

try:
    for screen in schedule:

        # show a short info screen whenever a new block (attribute) starts
        if screen["block_changed"]:
            whispy.ui.InfoWindow(f"Please rate the {screen['attribute']} next.")

        # open the rating screen (the first one owns the shared window; the
        # rest reuse it via parent=host). The one combined config carries both
        # the `ui:` layout and the `attributes:` rating scales.
        mushra_like = whispy.ui.DragAndDropMUSHRA(
            screen=screen,
            stimuli_handler=handler,
            drag_and_drop_mushra=cfg,
            parent=host)
        if host is None:
            host = mushra_like

        # accumulate the results (one row per rated stimulus)
        results = mushra_like.get_results(results)
finally:
    # close the window after the last trial so everybody knows it's over
    if host is not None:
        host.close()

## Thank the participant

The experiment is over. Run this cell to show the participant a **thank-you
screen**; its message lives in [`configs/thanks.yml`](../configs/thanks.yml)
(markdown supported), so you can reword it without editing the Python.

In [ ]:
from whispy.ui import InfoWindow

# The thank-you message lives in configs/thanks.yml (markdown supported); reword
# it there without editing this cell.
thanks = read_config(Path('..') / 'configs' / 'thanks.yml')['ui']
InfoWindow(thanks['message'], fullscreen=thanks.get('fullscreen', True), blocking=True)

## Results

`results` is in **long format** — one row per rated stimulus — combining the
screen metadata (block/section/attribute names, reference, test id) with the
decoded `rating`.

In [ ]:
results

## Save the results

Save the results to a CSV in a `results/` folder (created next to this notebook).
The file name always carries a **timestamp**. If a `participant_id` is in scope
(set by a consent block earlier in the notebook) it is included
(`<name>_<id>_<timestamp>.csv`); otherwise an iterating fallback number is used
(`<name>_<NNN>_<timestamp>.csv`). Existing files are never overwritten.

In [ ]:
from whispy.utils import save_results

# `participant_id` is picked up automatically if a consent block earlier in this
# notebook set it; otherwise it is None and the file name uses an iterating
# fallback number.
participant_id = globals().get('participant_id')
results_path = save_results(results, 'drag_and_drop_mushra', participant_id=participant_id)
print('saved results to', results_path.resolve())

## Plot the results

Create a scatterplot per stimulus highlighting the mean ratings and/or plot the classic MUSHRA summary plot with a 95% confidence intervall

In [ ]:
from whispy.utils import Plotting as plt

plt().plot_mushra_mean_ratings(results)
plt().plot_mushra_summary_distribution(results)